In [1]:
# 03 - YOLO11-cls Transfer Learning
# Same flow as 02_cnn_baseline: grid search -> save config -> train final model -> evaluate on val and test
# Model construction comes from src/models/yolo_train.py (RULEBOOK Section 6 - YOLO has its own API)

import sys
sys.path.insert(0, '..')

import os
import yaml
import torch

from src.models.yolo_train import (
    train_yolo,
    get_yolo_predictions,
    collect_image_paths,
    copy_best_weights,
)
from src.utils.metrics import show_confusion_matrix, show_classification_report

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


In [3]:
# YOLO classification expects a DIRECTORY (not a YAML file) containing
# train/ and val/ subfolders, each with one subfolder per class.
# data/processed/ already has exactly this structure - no conversion needed.

data_dir = "../data/processed"

for split in ["train", "val", "test"]:
    p = os.path.join(data_dir, split)
    if not os.path.isdir(p):
        print(p, "not found, run 01_data_prep.ipynb first")

In [4]:
# class names in the order YOLO will assign indices (alphabetical)
class_names = sorted(os.listdir(os.path.join(data_dir, "train")))
num_classes = len(class_names)
print("Classes:", class_names)

Classes: ['healthy', 'low_tread', 'sidewall_damaged', 'uneven_wear', 'zero_tread']


In [5]:
# grid search - try a few combinations, keep the best one based on val top-1 accuracy
# NOTE: transfer learning uses a SMALLER learning-rate range and FEWER epochs
# than a from-scratch CNN (see RULEBOOK Section 6)

learning_rates = [0.001, 0.0001]
batch_sizes = [16, 32]

best_val_acc = 0
best_settings = None

for lr in learning_rates:
    for bs in batch_sizes:
        print("Trying lr:", lr, "batch_size:", bs)

        _, results = train_yolo(
            data_dir=data_dir,
            model_name="yolo11n-cls.pt",
            epochs=5,                 # short run per combination
            imgsz=224,
            batch=bs,
            lr0=lr,
            freeze=10,                # freeze backbone for transfer learning
            project="../results/yolo11n",
            name=f"lr{lr}_bs{bs}",
        )

        # Ultralytics stores validation top-1 accuracy in results.results_dict
        metrics = results.results_dict
        val_acc = metrics.get("metrics/accuracy_top1", 0)
        print("  val top-1 acc:", round(val_acc, 3))

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_settings = {"learning_rate": lr, "batch_size": bs}

print()
print("Best settings:", best_settings, "with val acc:", round(best_val_acc, 3))

Trying lr: 0.001 batch_size: 16
New https://pypi.org/project/ultralytics/8.4.163 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.162  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3060, 12288MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../data/processed, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=5, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.

In [6]:
# save the winning settings to the config file automatically
config = {
    "model": "yolo11n-cls.pt",
    "image_size": 224,
    "batch_size": best_settings["batch_size"],
    "epochs": 50,                 # fewer than CNN's 40? No - YOLO11n-cls is smaller;
    "learning_rate": best_settings["learning_rate"],
    "freeze": 10,
    "num_classes": num_classes,
    "class_names": class_names,
}

with open('../configs/yolo11n_config.yaml', 'w') as f:
    yaml.dump(config, f)

print("Saved to configs/yolo11n_config.yaml:", config)

Saved to configs/yolo11n_config.yaml: {'model': 'yolo11n-cls.pt', 'image_size': 224, 'batch_size': 32, 'epochs': 50, 'learning_rate': 0.001, 'freeze': 10, 'num_classes': 5, 'class_names': ['healthy', 'low_tread', 'sidewall_damaged', 'uneven_wear', 'zero_tread']}


In [7]:
# load settings back from the config file
with open('../configs/yolo11n_config.yaml', 'r') as f:
    config = yaml.safe_load(f)

print("Training with:", config)

Training with: {'batch_size': 32, 'class_names': ['healthy', 'low_tread', 'sidewall_damaged', 'uneven_wear', 'zero_tread'], 'epochs': 50, 'freeze': 10, 'image_size': 224, 'learning_rate': 0.001, 'model': 'yolo11n-cls.pt', 'num_classes': 5}


In [8]:
# build the final model - fresh from pretrained weights
os.makedirs("../results/yolo11n", exist_ok=True)

model, results = train_yolo(
    data_dir=data_dir,
    model_name=config["model"],
    epochs=config["epochs"],
    imgsz=config["image_size"],
    batch=config["batch_size"],
    lr0=config["learning_rate"],
    freeze=config["freeze"],
    project="../results/yolo11n",
    name="train",
)

print("Training complete")
print("Results saved to:", results.save_dir)

New https://pypi.org/project/ultralytics/8.4.163 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.162  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3060, 12288MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../data/processed, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-cl

In [9]:
# copy best.pt from Ultralytics' run folder to results/yolo/model.pt
# (so the naming is consistent with results/cnn/model.pt and results/resnet18/model.pt)
copy_best_weights(
    run_dir=os.path.join("../results/yolo11n", "train"),
    dest_path="../results/yolo11n/model.pt",
)

No best.pt found at ../results/yolo11n\train\weights\best.pt


False

In [10]:
# evaluate on VAL set using the same metrics functions as every other model
val_paths = collect_image_paths(os.path.join(data_dir, "val"), class_names)
print("Val images:", len(val_paths))

val_true, val_pred = get_yolo_predictions(model, val_paths, class_names)

print("VAL SET RESULTS")
show_confusion_matrix(val_true, val_pred, class_names)
show_classification_report(val_true, val_pred, class_names)

Val images: 60
VAL SET RESULTS
Confusion matrix
(rows = actual, columns = predicted)
Classes: ['healthy', 'low_tread', 'sidewall_damaged', 'uneven_wear', 'zero_tread']
[[19  2  0  0  1]
 [ 4  5  0  1  1]
 [ 1  0  7  0  0]
 [ 0  2  0  5  1]
 [ 0  1  0  1  9]]
                  precision    recall  f1-score   support

         healthy       0.79      0.86      0.83        22
       low_tread       0.50      0.45      0.48        11
sidewall_damaged       1.00      0.88      0.93         8
     uneven_wear       0.71      0.62      0.67         8
      zero_tread       0.75      0.82      0.78        11

        accuracy                           0.75        60
       macro avg       0.75      0.73      0.74        60
    weighted avg       0.75      0.75      0.75        60



In [11]:
# evaluate on TEST set - final, honest, one-time score
test_paths = collect_image_paths(os.path.join(data_dir, "test"), class_names)
print("Test images:", len(test_paths))

test_true, test_pred = get_yolo_predictions(model, test_paths, class_names)

print("TEST SET RESULTS (final)")
show_confusion_matrix(test_true, test_pred, class_names)
show_classification_report(test_true, test_pred, class_names)

Test images: 67
TEST SET RESULTS (final)
Confusion matrix
(rows = actual, columns = predicted)
Classes: ['healthy', 'low_tread', 'sidewall_damaged', 'uneven_wear', 'zero_tread']
[[21  0  1  0  1]
 [ 2  5  0  2  3]
 [ 1  0  8  0  0]
 [ 0  2  0  3  5]
 [ 0  1  0  2 10]]
                  precision    recall  f1-score   support

         healthy       0.88      0.91      0.89        23
       low_tread       0.62      0.42      0.50        12
sidewall_damaged       0.89      0.89      0.89         9
     uneven_wear       0.43      0.30      0.35        10
      zero_tread       0.53      0.77      0.62        13

        accuracy                           0.70        67
       macro avg       0.67      0.66      0.65        67
    weighted avg       0.70      0.70      0.69        67



In [12]:
# save test results to a text file - small file, goes in git
from sklearn.metrics import classification_report

report_text = classification_report(test_true, test_pred, target_names=class_names)

os.makedirs("../results/yolo11n", exist_ok=True)

with open("../results/yolo11n/metrics.txt", "w") as f:
    f.write("Settings used: " + str(config) + "\n\n")
    f.write("Test set results:\n")
    f.write(report_text)

print("Saved to results/yolo11n/metrics.txt")

Saved to results/yolo11n/metrics.txt
